In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [
  (1, "Alice",  "Engineering", 95000, "2021-03-15", "F"),
  (2, "Bob",    "Marketing",   72000, "2019-07-01", "M"),
  (3, "Carol",  "Engineering", 88000, "2020-11-20", "F"),
  (4, "David",  "HR",          61000, "2022-01-10", "M"),
  (5, "Eve",    "Marketing",   79000, "2018-05-25", "F"),
  (6, "Frank",  "Engineering", 102000,"2017-09-30", "M"),
  (7, "Grace",  "HR",          58000, "2023-02-14", "F"),
]

schema = StructType([
  StructField("id",         IntegerType(), False),
  StructField("name",       StringType(),  True),
  StructField("dept",       StringType(),  True),
  StructField("salary",     IntegerType(), True),
  StructField("join_date",  StringType(),  True),
  StructField("gender",     StringType(),  True),
])

df = spark.createDataFrame(data, schema)

In [0]:
display(df)

id,name,dept,salary,join_date,gender
1,Alice,Engineering,95000,2021-03-15,F
2,Bob,Marketing,72000,2019-07-01,M
3,Carol,Engineering,88000,2020-11-20,F
4,David,HR,61000,2022-01-10,M
5,Eve,Marketing,79000,2018-05-25,F
6,Frank,Engineering,102000,2017-09-30,M
7,Grace,HR,58000,2023-02-14,F


In [0]:
dept_data = [("Engineering", "New York"),
             ("Marketing",   "Chicago"),
             ("Finance",     "Dallas")]

df_dept = spark.createDataFrame(dept_data, ["dept", "city"])

In [0]:
df_dept.display()

dept,city
Engineering,New York
Marketing,Chicago
Finance,Dallas


In [0]:
df.join(df_dept, on="dept", how="inner").display()

dept,id,name,salary,join_date,gender,city
Engineering,1,Alice,95000,2021-03-15,F,New York
Marketing,2,Bob,72000,2019-07-01,M,Chicago
Engineering,3,Carol,88000,2020-11-20,F,New York
Marketing,5,Eve,79000,2018-05-25,F,Chicago
Engineering,6,Frank,102000,2017-09-30,M,New York


In [0]:
df.join(df_dept, on="dept", how="left").display()

dept,id,name,salary,join_date,gender,city
Engineering,1,Alice,95000,2021-03-15,F,New York
Marketing,2,Bob,72000,2019-07-01,M,Chicago
Engineering,3,Carol,88000,2020-11-20,F,New York
HR,4,David,61000,2022-01-10,M,null
Marketing,5,Eve,79000,2018-05-25,F,Chicago
Engineering,6,Frank,102000,2017-09-30,M,New York
HR,7,Grace,58000,2023-02-14,F,null


In [0]:
df.join(df_dept, on="dept", how="right").display()
 # left anti

dept,id,name,salary,join_date,gender,city
Engineering,6,Frank,102000,2017-09-30,M,New York
Engineering,3,Carol,88000,2020-11-20,F,New York
Engineering,1,Alice,95000,2021-03-15,F,New York
Marketing,5,Eve,79000,2018-05-25,F,Chicago
Marketing,2,Bob,72000,2019-07-01,M,Chicago
Finance,null,null,null,null,null,Dallas


In [0]:
df.join(df_dept, on="dept", how="full").display()
  

dept,id,name,salary,join_date,gender,city
Engineering,1,Alice,95000,2021-03-15,F,New York
Marketing,2,Bob,72000,2019-07-01,M,Chicago
Engineering,3,Carol,88000,2020-11-20,F,New York
HR,4,David,61000,2022-01-10,M,null
Marketing,5,Eve,79000,2018-05-25,F,Chicago
Engineering,6,Frank,102000,2017-09-30,M,New York
HR,7,Grace,58000,2023-02-14,F,null
Finance,null,null,null,null,null,Dallas


In [0]:
df.join(df_dept, on="dept", how="left_semi").display()


dept,id,name,salary,join_date,gender
Engineering,1,Alice,95000,2021-03-15,F
Marketing,2,Bob,72000,2019-07-01,M
Engineering,3,Carol,88000,2020-11-20,F
Marketing,5,Eve,79000,2018-05-25,F
Engineering,6,Frank,102000,2017-09-30,M


In [0]:
df.join(df_dept, on="dept", how="anti").show()

+----+---+-----+------+----------+------+
|dept| id| name|salary| join_date|gender|
+----+---+-----+------+----------+------+
|  HR|  4|David| 61000|2022-01-10|     M|
|  HR|  7|Grace| 58000|2023-02-14|     F|
+----+---+-----+------+----------+------+



In [0]:
df.join(df_dept, on=["dept", "city"], how="inner").show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8298393184961022>, line 1
----> 1 df.join(df_dept, on=["dept", "city"], how="inner").show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(
    903         child=self._

In [0]:
new_data = [
  (8, "Heidi", "Finance",    85000, "2024-01-05", "F"),
  (9, "Ivan",  "Engineering",91000, "2023-08-01", "M"),
]
df2 = spark.createDataFrame(new_data, schema)  

In [0]:
df_all = df.union(df2)
df_all.show()

+---+-----+-----------+------+----------+------+
| id| name|       dept|salary| join_date|gender|
+---+-----+-----------+------+----------+------+
|  1|Alice|Engineering| 95000|2021-03-15|     F|
|  2|  Bob|  Marketing| 72000|2019-07-01|     M|
|  3|Carol|Engineering| 88000|2020-11-20|     F|
|  4|David|         HR| 61000|2022-01-10|     M|
|  5|  Eve|  Marketing| 79000|2018-05-25|     F|
|  6|Frank|Engineering|102000|2017-09-30|     M|
|  7|Grace|         HR| 58000|2023-02-14|     F|
|  8|Heidi|    Finance| 85000|2024-01-05|     F|
|  9| Ivan|Engineering| 91000|2023-08-01|     M|
+---+-----+-----------+------+----------+------+



In [0]:
df_all = df.unionByName(df2)
df_all.show()

+---+-----+-----------+------+----------+------+
| id| name|       dept|salary| join_date|gender|
+---+-----+-----------+------+----------+------+
|  1|Alice|Engineering| 95000|2021-03-15|     F|
|  2|  Bob|  Marketing| 72000|2019-07-01|     M|
|  3|Carol|Engineering| 88000|2020-11-20|     F|
|  4|David|         HR| 61000|2022-01-10|     M|
|  5|  Eve|  Marketing| 79000|2018-05-25|     F|
|  6|Frank|Engineering|102000|2017-09-30|     M|
|  7|Grace|         HR| 58000|2023-02-14|     F|
|  8|Heidi|    Finance| 85000|2024-01-05|     F|
|  9| Ivan|Engineering| 91000|2023-08-01|     M|
+---+-----+-----------+------+----------+------+



In [0]:
df_all = df.unionAll(df2)
df_all.show()

+---+-----+-----------+------+----------+------+
| id| name|       dept|salary| join_date|gender|
+---+-----+-----------+------+----------+------+
|  1|Alice|Engineering| 95000|2021-03-15|     F|
|  2|  Bob|  Marketing| 72000|2019-07-01|     M|
|  3|Carol|Engineering| 88000|2020-11-20|     F|
|  4|David|         HR| 61000|2022-01-10|     M|
|  5|  Eve|  Marketing| 79000|2018-05-25|     F|
|  6|Frank|Engineering|102000|2017-09-30|     M|
|  7|Grace|         HR| 58000|2023-02-14|     F|
|  8|Heidi|    Finance| 85000|2024-01-05|     F|
|  9| Ivan|Engineering| 91000|2023-08-01|     M|
+---+-----+-----------+------+----------+------+

